# CMPE492 — HDTF Data Preparation (Stage 1)

**Purpose:** Prepare a 10-identity benchmark set from the HDTF dataset for multi-identity evaluation.

For each identity this notebook produces:
- **Raw video** (512×512, 25fps, 30s) → driving signal for LP-only baseline + audio source
- **Audio** (.wav, 16kHz mono) → input for MimicTalk
- **Reference frame** (.png, neutral pose) → source image for both pipelines

**Output structure:**
```
/content/hdtf_prepared/
├── raw/         id01.mp4 ... id10.mp4   (512×512, 25fps, 30s)
├── audio/       id01.wav ... id10.wav
└── references/  id01.png ... id10.png
```

This feeds Stage 2 (MimicTalk) and Stage 3 (LivePortrait), then the Batch Evaluation notebook.

---
**Run order:** 1 → 2 → 3 → 4 → 5

## Cell 1 — Setup

In [ ]:
import subprocess, sys, os

# Install tools
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'yt-dlp'], check=True)
print('✅ yt-dlp installed')

# Clone HDTF metadata
HDTF_META = '/content/HDTF_meta'
if not os.path.exists(HDTF_META):
    subprocess.run(f'git clone https://github.com/MRzzm/HDTF.git {HDTF_META}',
                   shell=True, capture_output=True)
print('✅ HDTF metadata cloned')

# Output dirs
BASE = '/content/hdtf_prepared'
for sub in ['raw', 'audio', 'references']:
    os.makedirs(f'{BASE}/{sub}', exist_ok=True)
print(f'✅ Output dirs ready: {BASE}')

# Config
N_IDENTITIES = 10
CLIP_SECONDS = 30
print(f'\nConfig: {N_IDENTITIES} identities, {CLIP_SECONDS}s clips')

## Cell 2 — Parse HDTF Metadata & Select Identities

In [ ]:
import glob, os, re

# HDTF provides url + timestamp files per category (e.g. RD_*, WDA_*, WRA_*)
url_files = sorted(glob.glob(f'{HDTF_META}/HDTF_dataset/*_video_url.txt'))
time_files = sorted(glob.glob(f'{HDTF_META}/HDTF_dataset/*_annotion_time.txt'))
print('URL files:', [os.path.basename(f) for f in url_files])

# Parse: each line is "name https://youtube.com/..."
def parse_urls(path):
    entries = {}
    with open(path) as f:
        for line in f:
            line = line.strip()
            if not line: continue
            parts = line.split()
            url = next((p for p in parts if 'http' in p), None)
            if url:
                name = parts[0]
                entries[name] = url
    return entries

# Parse timestamps: "name start_time end_time start_time end_time ..."
def parse_times(path):
    times = {}
    with open(path) as f:
        for line in f:
            line = line.strip()
            if not line: continue
            parts = line.split()
            name = parts[0]
            # collect timestamp pairs (hh:mm:ss format)
            ts = [p for p in parts[1:] if ':' in p]
            times[name] = ts
    return times

all_urls = {}
all_times = {}
for uf in url_files:
    all_urls.update(parse_urls(uf))
for tf in time_files:
    all_times.update(parse_times(tf))

print(f'\nTotal videos in metadata: {len(all_urls)}')

# Select N identities that have both URL and timestamps
candidates = [name for name in all_urls if name in all_times and all_times[name]]
print(f'Candidates with timestamps: {len(candidates)}')

# Pick diverse selection: spread across categories
selected = candidates[:N_IDENTITIES * 2]  # extra in case some fail to download
print(f'\nSelected pool ({len(selected)}): {selected[:N_IDENTITIES]}')
print('(extras kept as fallback if downloads fail)')

## Cell 3 — Download, Clip & Normalize (with resume)

In [ ]:
import subprocess, shlex, os, json, cv2, time

BASE = '/content/hdtf_prepared'
PROGRESS = f'{BASE}/progress.json'

# Resume support
if os.path.exists(PROGRESS):
    with open(PROGRESS) as f: completed = json.load(f)
else:
    completed = {}

def hms_to_sec(t):
    parts = [int(x) for x in t.split(':')]
    if len(parts) == 3: return parts[0]*3600 + parts[1]*60 + parts[2]
    if len(parts) == 2: return parts[0]*60 + parts[1]
    return parts[0]

def prepare_identity(name, url, timestamps, idx):
    id_tag = f'id{idx:02d}'
    raw_out = f'{BASE}/raw/{id_tag}.mp4'
    audio_out = f'{BASE}/audio/{id_tag}.wav'
    ref_out = f'{BASE}/references/{id_tag}.png'

    if id_tag in completed and all(os.path.exists(p) for p in [raw_out, audio_out, ref_out]):
        print(f'  ✓ {id_tag} ({name}) already done — skipping')
        return True

    print(f'  ⬇ {id_tag} ({name})')

    # Determine clip window from first timestamp pair
    if len(timestamps) >= 2:
        start = hms_to_sec(timestamps[0])
    else:
        start = 0
    duration = CLIP_SECONDS

    # Download full video (best mp4 <=720p to save space/time)
    tmp_dl = f'/tmp/{id_tag}_full.mp4'
    if not os.path.exists(tmp_dl):
        r = subprocess.run(
            f'yt-dlp -f "bestvideo[height<=720][ext=mp4]+bestaudio[ext=m4a]/best[height<=720]" '
            f'--merge-output-format mp4 -o {shlex.quote(tmp_dl)} {shlex.quote(url)}',
            shell=True, capture_output=True, text=True)
        if r.returncode != 0 or not os.path.exists(tmp_dl):
            print(f'    ⚠ download failed: {r.stderr[-150:]}')
            return False

    # Clip + normalize to 512x512, 25fps
    r = subprocess.run(
        f'ffmpeg -y -ss {start} -t {duration} -i {shlex.quote(tmp_dl)} '
        f'-vf "scale=512:512:force_original_aspect_ratio=increase,crop=512:512,fps=25" '
        f'-c:v libx264 -crf 18 -c:a aac -ar 16000 -ac 1 {shlex.quote(raw_out)}',
        shell=True, capture_output=True, text=True)
    if r.returncode != 0 or not os.path.exists(raw_out):
        print(f'    ⚠ clip failed: {r.stderr[-150:]}')
        return False

    # Extract audio
    subprocess.run(
        f'ffmpeg -y -i {shlex.quote(raw_out)} -vn -ac 1 -ar 16000 {shlex.quote(audio_out)}',
        shell=True, capture_output=True)

    # Extract reference frame (frame at 0.5s — usually mouth more neutral than frame 0)
    cap = cv2.VideoCapture(raw_out)
    cap.set(cv2.CAP_PROP_POS_FRAMES, 12)  # ~0.5s at 25fps
    ok, frame = cap.read()
    cap.release()
    if ok:
        cv2.imwrite(ref_out, frame)
    else:
        # fallback to frame 0
        subprocess.run(f'ffmpeg -y -i {shlex.quote(raw_out)} -frames:v 1 {shlex.quote(ref_out)}',
                       shell=True, capture_output=True)

    # Cleanup full download to save space
    if os.path.exists(tmp_dl): os.remove(tmp_dl)

    completed[id_tag] = {'name': name, 'url': url, 'start': start}
    with open(PROGRESS, 'w') as f: json.dump(completed, f, indent=2)
    print(f'    ✅ {id_tag} ready')
    return True

# Process until we have N_IDENTITIES successful
success_count = len([k for k in completed if os.path.exists(f'{BASE}/raw/{k}.mp4')])
idx = success_count + 1

for name in selected:
    if success_count >= N_IDENTITIES:
        break
    url = all_urls[name]
    ts = all_times.get(name, [])
    if prepare_identity(name, url, ts, idx):
        success_count += 1
        idx += 1

print(f'\n{"="*50}')
print(f'  ✅ Prepared {success_count}/{N_IDENTITIES} identities')
print(f'{"="*50}')

## Cell 4 — Verify & Preview

In [ ]:
import glob, os, cv2
import matplotlib.pyplot as plt

BASE = '/content/hdtf_prepared'

raw_vids = sorted(glob.glob(f'{BASE}/raw/*.mp4'))
audios   = sorted(glob.glob(f'{BASE}/audio/*.wav'))
refs     = sorted(glob.glob(f'{BASE}/references/*.png'))

print(f'Raw videos : {len(raw_vids)}')
print(f'Audio files: {len(audios)}')
print(f'References : {len(refs)}')

# Verify each video's properties
print('\n── Per-identity check ──')
import subprocess
for v in raw_vids:
    stem = os.path.splitext(os.path.basename(v))[0]
    probe = subprocess.run(
        f'ffprobe -v quiet -show_entries format=duration:stream=width,height,r_frame_rate '
        f'-of csv=p=0 {v}', shell=True, capture_output=True, text=True).stdout.strip().replace("\n", " ")
    has_audio = os.path.exists(f'{BASE}/audio/{stem}.wav')
    has_ref = os.path.exists(f'{BASE}/references/{stem}.png')
    print(f'  {stem}: {probe}  audio={has_audio} ref={has_ref}')

# Preview reference frames
n = len(refs)
if n > 0:
    cols = min(5, n)
    rows = (n + cols - 1) // cols
    plt.figure(figsize=(3*cols, 3*rows))
    for i, ref in enumerate(refs):
        img = cv2.cvtColor(cv2.imread(ref), cv2.COLOR_BGR2RGB)
        plt.subplot(rows, cols, i+1)
        plt.imshow(img)
        plt.title(os.path.basename(ref)); plt.axis('off')
    plt.tight_layout(); plt.show()

## Cell 5 — Package for Download

In [ ]:
import shutil, os
from google.colab import files

BASE = '/content/hdtf_prepared'

# Zip the whole prepared set
shutil.make_archive('/content/hdtf_prepared', 'zip', BASE)
sz = os.path.getsize('/content/hdtf_prepared.zip') // (1024*1024)
print(f'✅ Packaged: /content/hdtf_prepared.zip ({sz} MB)')
print('\nContents:')
print('  raw/        → driving videos for LP-only + audio source')
print('  audio/      → input for MimicTalk (Stage 2)')
print('  references/ → source images for both pipelines')

files.download('/content/hdtf_prepared.zip')
print('\n✅ Stage 1 complete. Next: Stage 2 (MimicTalk per identity)')